# HWP 파서 동작 과정 (UDFP 패키지 사용)

`udfp` 패키지의 파서 모듈을 import하여 HWP 파싱 과정을 단계별로 확인합니다.

Raw 바이트 조작 버전은 `hwp_parser_walkthrough.ipynb` 참고.

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKSPACE = os.path.join(PROJECT_ROOT, "workspace")
print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"워크스페이스:  {WORKSPACE}")
print(f"HWP 파일 목록:")
for f in sorted(os.listdir(WORKSPACE)):
    if f.endswith(".hwp"):
        print(f"  {f}")

In [ ]:
# 분석할 파일 선택 (여기를 바꾸면 다른 파일로 실험 가능)
TARGET = os.path.join(WORKSPACE, "10_인라인혼합서식.hwp")
print(f"분석 대상: {os.path.basename(TARGET)}")
print(f"파일 크기: {os.path.getsize(TARGET):,} bytes")

---
## 1단계: OLE2 컨테이너 열기

HWP 5.x는 Microsoft Compound File (OLE2) 컨테이너입니다.  
내부에 `FileHeader`, `DocInfo`, `BodyText/Section0` 등의 스트림이 들어 있습니다.

In [ ]:
from udfp.parsers.hwp.ole import OleReader

ole = OleReader.open(TARGET)

print("=== OLE2 스트림 목록 ===")
for s in ole.stream_names():
    info = ole.stream_info(s)
    raw = ole.read_raw(s)
    print(f"  {'/'.join(s):30s}  compressed={info.compressed}  raw_size={len(raw):>6,} bytes")

print(f"\n압축 사용: {ole._compressed}")
print(f"체크섬:    {ole.original_container.checksum}")

In [ ]:
import struct

# FileHeader 직접 확인
fh_raw = ole.read_raw(["FileHeader"])
print(f"FileHeader 크기: {len(fh_raw)} bytes")
print(f"시그니처:  {fh_raw[:32]}")

flags = struct.unpack_from('<I', fh_raw, 36)[0]
print(f"\nflags (offset 36): 0x{flags:08X}")
print(f"  bit0 (압축):      {bool(flags & 0x01)}")
print(f"  bit1 (암호화):    {bool(flags & 0x02)}")
print(f"  bit2 (배포제한):  {bool(flags & 0x04)}")
print(f"  bit3 (스크립트):  {bool(flags & 0x08)}")

---
## 2단계: HWPTAG 레코드 디코딩

DocInfo와 BodyText 스트림은 모두 동일한 형식의 **HWPTAG 레코드** 체인으로 구성됩니다.  
각 레코드는 4바이트 헤더: `tag_id(10bit) | level(10bit) | size(12bit)`

In [ ]:
from udfp.parsers.hwp.records import iter_records, HWPTAG_BEGIN

# 태그 이름 매핑 (주요 태그만)
TAG_NAMES = {
    16: "DOCUMENT_PROPERTIES", 17: "ID_MAPPINGS", 18: "BIN_DATA",
    19: "FACE_NAME", 20: "BORDER_FILL", 21: "CHAR_SHAPE",
    22: "TAB_DEF", 23: "NUMBERING", 24: "BULLET",
    25: "PARA_SHAPE", 26: "STYLE", 27: "DOC_DATA",
    66: "PARA_HEADER", 67: "PARA_TEXT", 68: "PARA_CHAR_SHAPE",
    69: "PARA_LINE_SEG", 70: "PARA_RANGE_TAG",
    71: "CTRL_HEADER", 72: "LIST_HEADER", 73: "PAGE_DEF",
    77: "TABLE",
}

def tag_name(tid):
    return TAG_NAMES.get(tid, f"TAG_{tid}")

In [ ]:
# DocInfo 스트림 레코드 목록
docinfo_raw = ole.read_stream(["DocInfo"])
print(f"DocInfo 스트림: {len(docinfo_raw):,} bytes (압축 해제 후)\n")

docinfo_records = list(iter_records(docinfo_raw))
print(f"레코드 수: {len(docinfo_records)}\n")

# 태그별 카운트
from collections import Counter
tag_counts = Counter(tag_name(r.tag_id) for r in docinfo_records)
print("태그별 카운트:")
for name, cnt in tag_counts.most_common():
    print(f"  {name:25s} × {cnt}")

In [ ]:
# BodyText/Section0 레코드 목록
body_raw = ole.read_stream(["BodyText", "Section0"])
print(f"BodyText/Section0: {len(body_raw):,} bytes (압축 해제 후)\n")

body_records = list(iter_records(body_raw))
print(f"레코드 수: {len(body_records)}\n")

# 레코드 트리 시각화 (들여쓰기로 level 표현)
print("레코드 트리:")
for rec in body_records:
    indent = "  " * rec.level
    name = tag_name(rec.tag_id)
    extra = ""
    if rec.tag_id == 71:  # CTRL_HEADER
        from udfp.parsers.hwp.records import ctrl_id_from_payload
        extra = f"  ctrl_id='{ctrl_id_from_payload(rec.payload)}'"
    print(f"{indent}[L{rec.level}] {name:20s} payload={len(rec.payload):>4}B  offset={rec.offset}{extra}")

---
## 3단계: DocInfo 파싱 — GlobalResources 생성

DocInfo에서 CharShape, ParaShape, FaceName, Style 등을 인덱싱합니다.  
이후 BodyText 파싱 시 `char_shape_id=3`처럼 인덱스로 참조합니다.

In [ ]:
from udfp.parsers.hwp.doc_info import parse_doc_info

info = parse_doc_info(docinfo_raw)

print(f"=== FaceNames ({len(info.face_names)}개) ===")
for i, fn in enumerate(info.face_names):
    print(f"  [{i:2d}] {fn}")

In [ ]:
print(f"=== CharShapes ({len(info.char_shapes)}개) ===")
for i, cs in enumerate(info.char_shapes):
    face_id = cs.get('hangul_face_id', '?')
    font = info.face_names[face_id] if isinstance(face_id, int) and face_id < len(info.face_names) else '?'
    size = cs.get('font_size_pt', '?')
    attrs = []
    if cs.get('bold'): attrs.append('B')
    if cs.get('italic'): attrs.append('I')
    if cs.get('underline'): attrs.append('U')
    if cs.get('strikethrough'): attrs.append('S')
    if cs.get('color'): attrs.append(f"c={cs['color']}")
    attr_str = ', '.join(attrs) if attrs else 'plain'
    print(f"  [{i:2d}] font={font}, size={size}pt, [{attr_str}]")

In [ ]:
print(f"=== ParaShapes ({len(info.para_shapes)}개) ===")
for i, ps in enumerate(info.para_shapes):
    align = ps.get('alignment', '?')
    ls = ps.get('line_spacing_hwp', 0)
    ls_type = ps.get('line_spacing_type', '?')
    print(f"  [{i:2d}] align={align:8s}  line_spacing={ls} ({ls_type})")

In [ ]:
print(f"=== Styles ({len(info.style_names)}개) ===")
gr = info.global_resources
for idx, sdef in gr.styles.items():
    fmt_info = ""
    if sdef.format:
        parts = []
        if sdef.format.alignment: parts.append(f"align={sdef.format.alignment}")
        if sdef.format.bold: parts.append("bold")
        if sdef.format.font_size: parts.append(f"size={sdef.format.font_size}")
        fmt_info = f"  fmt=[{', '.join(parts)}]" if parts else ""
    print(f"  [{idx:>2s}] {sdef.name:20s} type={sdef.style_type or '?':12s}{fmt_info}")

---
## 4단계: PARA_TEXT 텍스트 추출

PARA_TEXT 페이로드는 UTF-16LE 문자와 제어 코드가 혼재합니다.  
제어 코드 규칙:
- `0x0000, 0x0009, 0x000A, 0x000D` → 2바이트만 소비  
- 그 외 `≤ 0x001F` → 인라인 오브젝트, 16바이트 소비

In [ ]:
from udfp.parsers.hwp.records import _PT_CTRL_2BYTE

# 첫 번째 PARA_TEXT 레코드를 찾아서 상세 분석
para_texts = [r for r in body_records if r.tag_id == 67]  # HWPTAG_PARA_TEXT
print(f"PARA_TEXT 레코드 수: {len(para_texts)}\n")

for pt_idx, pt_rec in enumerate(para_texts):
    pay = pt_rec.payload
    print(f"--- PARA_TEXT[{pt_idx}] (payload {len(pay)} bytes) ---")
    
    # 바이트 단위로 디코딩 과정 추적
    off = 0
    chars = []
    trace = []
    while off + 2 <= len(pay):
        code = struct.unpack_from('<H', pay, off)[0]
        if code > 0x001F:
            chars.append(chr(code))
            trace.append(f"  off={off:3d}: U+{code:04X} → '{chr(code)}'")
            off += 2
        elif code in _PT_CTRL_2BYTE:
            trace.append(f"  off={off:3d}: CTRL 0x{code:04X} (2B skip)")
            off += 2
        else:
            trace.append(f"  off={off:3d}: CTRL 0x{code:04X} (16B inline object)")
            off += 16
    
    text = ''.join(chars)
    print(f"  추출 텍스트: \"{text}\"")
    print(f"  문자 수: {len(chars)}")
    
    # 처음 몇 개 trace만 표시
    if len(trace) > 12:
        for t in trace[:6]: print(t)
        print(f"  ... ({len(trace) - 12}개 생략) ...")
        for t in trace[-6:]: print(t)
    else:
        for t in trace: print(t)
    print()

---
## 5단계: PARA_CHAR_SHAPE → 인라인 분할

PCS(PARA_CHAR_SHAPE) 레코드는 `(문자위치, char_shape_id)` 쌍의 배열입니다.  
같은 CharShape 구간을 하나의 `TextInline`으로 묶어 서식을 전파합니다.

In [ ]:
from udfp.parsers.hwp.records import (
    HWPTAG_PARA_HEADER, HWPTAG_PARA_TEXT,
    HWPTAG_PARA_CHAR_SHAPE, HWPTAG_PARA_LINE_SEG
)

para_idx = 0
for i, rec in enumerate(body_records):
    if rec.tag_id != HWPTAG_PARA_HEADER:
        continue
    
    children = []
    for j in range(i+1, len(body_records)):
        if body_records[j].level <= rec.level:
            break
        children.append(body_records[j])
    
    pt = next((c for c in children if c.tag_id == HWPTAG_PARA_TEXT), None)
    pcs = next((c for c in children if c.tag_id == HWPTAG_PARA_CHAR_SHAPE), None)
    
    if pt and pcs:
        print(f"=== 단락 {para_idx} ===")
        
        entries = []
        for k in range(0, len(pcs.payload) - 7, 8):
            pos, cs_id = struct.unpack_from('<II', pcs.payload, k)
            entries.append((pos, cs_id))
        
        print(f"PCS 엔트리 ({len(entries)}개):")
        for pos, cs_id in entries:
            cs = info.char_shapes[cs_id] if cs_id < len(info.char_shapes) else {}
            attrs = []
            if cs.get('bold'): attrs.append('B')
            if cs.get('italic'): attrs.append('I')
            if cs.get('underline'): attrs.append('U')
            attr_str = ','.join(attrs) if attrs else 'plain'
            print(f"  pos={pos:3d} → char_shape[{cs_id}] [{attr_str}]")
        
        from udfp.parsers.hwp.body import _build_inlines
        inlines = _build_inlines(pt.payload, pcs.payload, info.char_shapes, info.face_names)
        print(f"\n생성된 TextInline ({len(inlines)}개):")
        for il in inlines:
            attrs = []
            if il.bold: attrs.append('B')
            if il.italic: attrs.append('I')
            if il.underline: attrs.append('U')
            if il.strikethrough: attrs.append('S')
            attr_str = ','.join(attrs) if attrs else 'plain'
            print(f"  \"{il.text}\" [{attr_str}] font={il.font_name} size={il.font_size}")
        print()
    
    para_idx += 1
    if para_idx >= 4:
        break

---
## 6단계: PARA_HEADER 메타데이터

PARA_HEADER에서 `para_shape_id`(offset 8)와 `style_id`(offset 10)를 읽어  
BlockFormat과 HeadingBlock 감지에 사용합니다.

In [ ]:
para_headers = [r for r in body_records if r.tag_id == HWPTAG_PARA_HEADER]
print(f"PARA_HEADER 수: {len(para_headers)}\n")

for idx, ph in enumerate(para_headers):
    pay = ph.payload
    print(f"--- PARA_HEADER[{idx}] (level={ph.level}, payload={len(pay)}B) ---")
    
    if len(pay) >= 4:
        char_cnt_raw = struct.unpack_from('<I', pay, 0)[0]
        msb = bool(char_cnt_raw & 0x80000000)
        char_cnt = char_cnt_raw & 0x3FFFFFFF
        print(f"  charCnt: {char_cnt} (MSB={msb})")
    
    if len(pay) >= 6:
        ctrl_mask = struct.unpack_from('<H', pay, 4)[0]
        print(f"  controlMask: 0x{ctrl_mask:04X}")
    
    if len(pay) >= 10:
        ps_id = struct.unpack_from('<H', pay, 8)[0]
        ps = info.para_shapes[ps_id] if ps_id < len(info.para_shapes) else {}
        print(f"  para_shape_id: {ps_id} → align={ps.get('alignment', '?')}")
    
    if len(pay) >= 11:
        style_id = pay[10]
        style_name = info.style_names[style_id] if style_id < len(info.style_names) else '?'
        print(f"  style_id: {style_id} → '{style_name}'")
        from udfp.parsers.hwp.body import _heading_level_from_style
        hl = _heading_level_from_style(style_name)
        if hl:
            print(f"  → HeadingBlock level={hl}")
    print()

---
## 7단계: 전체 파싱 결과 — UdfpDocument IR

`parse_hwp()`를 호출하여 최종 IR을 확인합니다.

In [ ]:
from udfp.parsers.hwp.parse import parse_hwp

doc = parse_hwp(TARGET)

print(f"source_format: {doc.source_format}")
print(f"udfp version:  {doc.udfp}")
print(f"blocks:        {len(doc.blocks)}")
print(f"verbatim:      {len(doc.verbatim.blocks) if doc.verbatim else 0} blocks")
print(f"checksum:      {doc.conversion_trace.checksum}")
print(f"parsed_at:     {doc.conversion_trace.parsed_at}")

In [ ]:
print("=== 시맨틱 블록 목록 ===")
for b in doc.blocks:
    t = b.type
    if t == 'paragraph':
        for il in b.inlines:
            attrs = []
            if getattr(il, 'bold', None): attrs.append('B')
            if getattr(il, 'italic', None): attrs.append('I')
            if getattr(il, 'underline', None): attrs.append('U')
            if getattr(il, 'strikethrough', None): attrs.append('S')
            if getattr(il, 'color', None): attrs.append(f"c={il.color}")
            attr_str = ','.join(attrs) if attrs else 'plain'
            print(f"  [para] id={b.id}  \"{il.text}\" [{attr_str}]")
        if not b.inlines:
            print(f"  [para] id={b.id}  (빈 단락)")
    elif t == 'heading':
        print(f"  [h{b.level}]  id={b.id}  \"{b.text}\"")
    elif t == 'table':
        print(f"  [table] id={b.id}  rows={len(b.rows)}")
        for ri, row in enumerate(b.rows):
            for ci, cell in enumerate(row.cells):
                cell_text = ''
                for cb in cell.content:
                    if hasattr(cb, 'inlines'):
                        cell_text = ' '.join(getattr(il, 'text', '') for il in cb.inlines)
                print(f"         [{ri},{ci}] span=({cell.row_span},{cell.col_span}) \"{cell_text}\"")
    elif t == 'unknown':
        print(f"  [unknown] id={b.id}  {b.description}")
    else:
        print(f"  [{t}] id={b.id}")

---
## 8단계: Verbatim 레이어 확인

각 시맨틱 블록은 `verbatim_ref`로 원본 바이너리와 연결됩니다.  
이 Verbatim 데이터가 있어야 Seed Patch 모드에서 무손실 재생성이 가능합니다.

In [ ]:
import base64

print("=== Verbatim 블록 ===")
for vid, vb in doc.verbatim.blocks.items():
    raw_len = len(base64.b64decode(vb.raw_bytes)) if vb.raw_bytes else 0
    decoded_keys = list(vb.decoded.keys()) if vb.decoded else []
    print(f"  {vid}: tag={tag_name(vb.raw_tag_id) if vb.raw_tag_id else '?'}, "
          f"level={vb.level}, raw={raw_len}B, decoded_keys={decoded_keys}")

print(f"\n=== Section Streams (Seed Patch 복원용) ===")
for name, b64 in doc.verbatim.section_streams.items():
    data = base64.b64decode(b64)
    print(f"  {name}: {len(data):,} bytes")

In [ ]:
# 시맨틱 블록 ↔ Verbatim 양방향 추적 예시
print("=== 블록 ↔ Verbatim 매핑 ===")
for b in doc.blocks:
    vref = getattr(b, 'verbatim_ref', None)
    if vref and vref in doc.verbatim.blocks:
        vb = doc.verbatim.blocks[vref]
        
        if b.type == 'paragraph':
            text = ' '.join(getattr(il, 'text', '') for il in b.inlines)[:40]
            semantic = f'"{ text}..."' if len(text) >= 40 else f'"{text}"'
        elif b.type == 'heading':
            semantic = f'H{b.level}: "{b.text}"'
        else:
            semantic = b.type
        
        raw_len = len(base64.b64decode(vb.raw_bytes)) if vb.raw_bytes else 0
        has_pt = bool(vb.decoded and vb.decoded.get('pt_bytes'))
        has_pcs = bool(vb.decoded and vb.decoded.get('pcs_bytes'))
        
        print(f"  {b.id} → {vref}")
        print(f"    시맨틱: {semantic}")
        print(f"    원본:   PH={raw_len}B, has_PT={has_pt}, has_PCS={has_pcs}")
        print()

---
## 9단계: JSON IR 출력

최종 `UdfpDocument`를 JSON으로 직렬화한 결과입니다.

In [ ]:
import json

ir_json = doc.model_dump(by_alias=True, exclude_none=True)

display_json = json.loads(json.dumps(ir_json))

if 'verbatim' in display_json and 'blocks' in display_json['verbatim']:
    for vid, vb in display_json['verbatim']['blocks'].items():
        if 'rawBytes' in vb and len(vb['rawBytes']) > 30:
            vb['rawBytes'] = vb['rawBytes'][:30] + '...'
        if 'decoded' in vb:
            for k in ('ptBytes', 'pcsBytes', 'plsBytes'):
                if k in vb['decoded'] and vb['decoded'][k] and len(vb['decoded'][k]) > 30:
                    vb['decoded'][k] = vb['decoded'][k][:30] + '...'

if 'verbatim' in display_json and 'sectionStreams' in display_json['verbatim']:
    for k in display_json['verbatim']['sectionStreams']:
        v = display_json['verbatim']['sectionStreams'][k]
        display_json['verbatim']['sectionStreams'][k] = v[:40] + f'... ({len(v)} chars)'

if 'verbatim' in display_json and 'globalResources' in display_json['verbatim']:
    gr = display_json['verbatim']['globalResources']
    for key in ('charShapes', 'paraShapes'):
        if key in gr and len(gr[key]) > 3:
            keys = list(gr[key].keys())
            gr[key] = {k: gr[key][k] for k in keys[:3]}
            gr[key]['...'] = f'({len(keys) - 3}개 더)'

print(json.dumps(display_json, ensure_ascii=False, indent=2))

---
## 10단계: 다른 파일로 실험

위의 `TARGET`을 바꿔서 다른 파일도 분석해 보세요.

In [ ]:
files = [
    "01_순수텍스트.hwp",
    "02_글자서식_볼드이탤릭밑줄.hwp",
    "05_표_셀내용.hwp",
    "09_헤딩_H1H2.hwp",
    "10_인라인혼합서식.hwp",
]

for fname in files:
    fpath = os.path.join(WORKSPACE, fname)
    if not os.path.exists(fpath):
        continue
    d = parse_hwp(fpath)
    block_types = [b.type for b in d.blocks]
    from collections import Counter as C
    tc = C(block_types)
    summary = ', '.join(f"{t}×{n}" for t, n in tc.most_common())
    print(f"{fname:40s}  blocks={len(d.blocks):2d}  [{summary}]")

In [ ]:
ole.close()
print("OLE reader 닫힘. 노트북 완료.")